# IEEE Taxonomy & Thesaurus Knowledge Graph Converter

Notebook ini berfungsi untuk mengonversi data **IEEE Taxonomy (Januari 2026)** dan **IEEE Thesaurus (Januari 2026)** dari format PDF menjadi format Linked Data (Turtle/RDF) menggunakan standar **SKOS (Simple Knowledge Organization System)**.

### Fitur Utama:
1. **Sistematis**: Pemrosesan dibagi menjadi fase-fase yang jelas.
2. **Pembersihan Robust**: Menangani deretan titik (leader dots) dan whitespace yang berantakan dari PDF.
3. **Kompatibilitas Pipeline**: Menambahkan `skos:prefLabel` dan `skos:altLabel` agar dapat dibaca oleh `TaxonomyEngine`.
4. **Standar SKOS**: Menggunakan `skos:broader`, `skos:narrower`, dan `skos:Concept` secara konsisten.

## 0. Persiapan & Konfigurasi

In [ ]:
import re
import os
import urllib.parse
from pathlib import Path
from collections import defaultdict, Counter
from PyPDF2 import PdfReader
from rdflib import Graph, Literal, RDF, URIRef, Namespace, RDFS
from rdflib.namespace import SKOS, OWL, XSD

# Konfigurasi Path
BASE_DIR = Path.cwd()
TAXONOMY_PDF = BASE_DIR / 'IEEE' / 'ieee-taxonomy-jan-2026.pdf'
THESAURUS_PDF = BASE_DIR / 'IEEE' / 'ieee-thesaurus-jan-2026.pdf'

# Namespace Definitions
IEEE_TAX_NS = Namespace("https://ieee-taxonomy.org/schema#")
IEEE_THES_NS = Namespace("https://ieee-thesaurus.org/schema#")

print(f"Working Directory: {os.getcwd()}")
print(f"Taxonomy Path: {TAXONOMY_PDF.exists()}")
print(f"Thesaurus Path: {THESAURUS_PDF.exists()}")

## 1. Helper Functions
Fungsi utilitas untuk membersihkan teks dan pembuatan URI yang standar.

In [ ]:
def clean_label(text):
    """Membersihkan label dari deretan titik dan spasi berlebih."""
    if not text: return ""
    # Hapus deretan titik (2 atau lebih) yang biasanya menyambung ke nomor halaman
    text = re.sub(r'\.\.+', '', text)
    # Hapus whitespace berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def make_uri(namespace, text):
    """Membuat URI yang aman dari sebuah label teks."""
    # Case-insensitive URI logic
    clean = clean_label(text).lower()
    clean = re.sub(r'[^a-z0-9]+', '_', clean).strip('_')
    safe_path = urllib.parse.quote(clean)
    return URIRef(f"{namespace}{safe_path}")

def extract_pdf_text(pdf_path, skip_pages=0):
    """Ekstraksi teks dari PDF dengan opsi skip halaman awal."""
    reader = PdfReader(pdf_path)
    full_text = []
    for i in range(skip_pages, len(reader.pages)):
        page_text = reader.pages[i].extract_text()
        if page_text:
            full_text.append(page_text)
    return "\n".join(full_text)

## 2. Fase 1: IEEE Taxonomy Extraction
Taxonomy IEEE menggunakan format hierarki berbasis titik (dots) di awal baris.

In [ ]:
print("Extracting IEEE Taxonomy...")
tax_text = extract_pdf_text(TAXONOMY_PDF)
tax_lines = tax_text.split('\n')

g_tax = Graph()
g_tax.bind("skos", SKOS)
g_tax.bind("ieee-tax", IEEE_TAX_NS)

parent_stack = []  # List of (uri, level)

for line in tax_lines:
    if not line.strip() or line.startswith('IEEE Taxonomy'): continue
    
    # Hitung level berdasarkan titik di awal
    level = len(line) - len(line.lstrip('.'))
    label = clean_label(line)
    
    if not label: continue
    
    uri = make_uri(IEEE_TAX_NS, label)
    
    # Tambahkan triple dasar
    g_tax.add((uri, RDF.type, SKOS.Concept))
    g_tax.add((uri, SKOS.prefLabel, Literal(label)))
    
    # Manage Hierarchy
    while parent_stack and parent_stack[-1][1] >= level:
        parent_stack.pop()
        
    if parent_stack:
        parent_uri, _ = parent_stack[-1]
        g_tax.add((uri, SKOS.broader, parent_uri))
        g_tax.add((parent_uri, SKOS.narrower, uri))
        
    parent_stack.append((uri, level))

print(f"Taxonomy extraction done. Total triples: {len(g_tax)}")

## 3. Fase 2: IEEE Thesaurus Extraction
Thesaurus menggunakan format keyword (BT, NT, RT, USE, UF) untuk mendefinisikan relasi.

In [ ]:
print("Extracting IEEE Thesaurus...")
thes_text = extract_pdf_text(THESAURUS_PDF, skip_pages=2)
thes_lines = thes_text.split('\n')

g_thes = Graph()
g_thes.bind("skos", SKOS)
g_thes.bind("ieee-thes", IEEE_THES_NS)

REL_RE = re.compile(r'^\s*(BT|NT|RT|USE|UF):\s*(.+?)\s*$')
SKIP_KEYWORDS = {'january', 'ieee thesaurus', 'version', 'work is licensed', 'created by'}

current_concept_uri = None
current_concept_label = None

for line in thes_lines:
    line_lower = line.lower().strip()
    if not line_lower or any(k in line_lower for k in SKIP_KEYWORDS): continue
    
    rel_match = REL_RE.match(line)
    if rel_match:
        if current_concept_uri:
            rel_type = rel_match.group(1)
            target_label = clean_label(rel_match.group(2))
            target_uri = make_uri(IEEE_THES_NS, target_label)
            
            # Pastikan target juga skos:Concept
            g_thes.add((target_uri, RDF.type, SKOS.Concept))
            g_thes.add((target_uri, SKOS.prefLabel, Literal(target_label)))
            
            if rel_type == 'BT':
                g_thes.add((current_concept_uri, SKOS.broader, target_uri))
            elif rel_type == 'NT':
                g_thes.add((current_concept_uri, SKOS.narrower, target_uri))
            elif rel_type == 'RT':
                g_thes.add((current_concept_uri, SKOS.related, target_uri))
            elif rel_type == 'USE':
                # current_label adalah non-preferred, target_label adalah preferred
                # Kita buat target_label punya altLabel current_label
                g_thes.add((target_uri, SKOS.altLabel, Literal(current_concept_label)))
            elif rel_type == 'UF':
                # current_label adalah preferred, target_label adalah non-preferred
                g_thes.add((current_concept_uri, SKOS.altLabel, Literal(target_label)))
    else:
        # Baris ini kemungkinan adalah term baru
        # Deteksi term: biasanya tidak dimulai dengan spasi banyak dan bukan keyword relasi
        words = line.strip().split()
        if words and words[0] not in ('BT', 'NT', 'RT', 'USE', 'UF'):
            current_concept_label = clean_label(line)
            current_concept_uri = make_uri(IEEE_THES_NS, current_concept_label)
            
            g_thes.add((current_concept_uri, RDF.type, SKOS.Concept))
            g_thes.add((current_concept_uri, SKOS.prefLabel, Literal(current_concept_label)))

print(f"Thesaurus extraction done. Total triples: {len(g_thes)}")

## 4. Fase 3: Export & Validasi
Menyimpan hasil ke dalam format `.ttl` yang akan digunakan oleh `TaxonomyEngine`.

In [ ]:
print("Saving Turtle files...")

g_tax.serialize(destination='ieee-taxonomy.ttl', format='turtle')
g_thes.serialize(destination='ieee-thesaurus.ttl', format='turtle')

print("Success!")
print(f"Files saved to: {os.getcwd()}")
print(f"Taxonomy: {len(g_tax)} triples")
print(f"Thesaurus: {len(g_thes)} triples")

### Verifikasi Singkat
Cek apakah `skos:prefLabel` dan `skos:Concept` sudah benar.

In [ ]:
from rdflib.plugins.sparql import prepareQuery

def test_graph(g, name):
    print(f"\n--- Testing {name} ---")
    q = prepareQuery("""
        SELECT (COUNT(?c) AS ?count) WHERE {
            ?c a skos:Concept .
            ?c skos:prefLabel ?l .
        }
    """, initNs={"skos": SKOS})
    
    for row in g.query(q):
        print(f"Concepts with prefLabel: {row.count}")

test_graph(g_tax, "Taxonomy")
test_graph(g_thes, "Thesaurus")